# Визуальная проверка спреда с backup (`backup1tb:spread-compacted`)

**Назначение:** быстрый sanity-check canary v1 parquet — графики `spread_long` / `spread_short` и latency (`okx` / `bybit` / `max`) за ~последние сутки для одной монеты.

**Контекст**

| | |
|---|---|
| Durable remote | `backup1tb:spread-compacted` |
| Формат | consolidated 5‑мин окна: `spread_YYYYMMDDTHHMMSSZ_….parquet` (все монеты в одном файле) |
| Схема | v1 (`spread_long`, `spread_short`, L1, …) — canary 2026-08-03/04 |
| Evidence | `docs/backup-validity-post-canary-early-stop-20260804.md` |

**Как запускать**

1. **VPS** — предпочтительно: rclone уже настроен (`/opt/rclone-1.74.4/rclone`), скачать окна в `CACHE_DIR`, затем читать parquet.
2. **Локальный Mac** — сначала один раз скачать нужные файлы (через VPS `rclone copy` + `scp`, или локальный rclone, если remote настроен), затем указать `CACHE_DIR` / `LOCAL_PARQUET_GLOB`.

Не трогает collector, systemd и remote (только list/copy).

## 1. Конфиг

Меняйте `BASE_COIN`, окно времени и пути кэша. По умолчанию: монета `0G` (есть в `bybit_okx_universe.csv` и в canary backup).

In [ ]:
from __future__ import annotations

import os
import re
import subprocess
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq

# --- что смотрим ---
BASE_COIN = "0G"  # есть в canary backup (и в bybit_okx_universe.csv); альтернатива: "BTC"
LOOKBACK_HOURS = 24  # относительно конца доступных данных на remote/cache
# Если задано — жёсткое календарное окно UTC (иначе выводится из имён файлов):
END_UTC: datetime | None = None  # например datetime(2026, 8, 4, 11, 45, tzinfo=timezone.utc)
START_UTC: datetime | None = None

# --- откуда читать ---
# После rclone copy / scp указывайте кэш. На VPS удобно /tmp/…
CACHE_DIR = Path(os.environ.get("SPREAD_BACKUP_CACHE", "/tmp/spread_backup_cache"))
# Уже скачанные файлы (абсолютный/относительный glob). Пример:
# LOCAL_PARQUET_GLOB = "/tmp/spread_backup_cache/spread_20260804T*.parquet"
LOCAL_PARQUET_GLOB: str | None = None

# На VPS локальные sent/compacted ещё могут лежать на диске (без rclone):
EXTRA_LOCAL_DIRS = [
    Path("/data/compacted"),
    Path("/data/compacted/sent"),
]

# --- rclone (VPS) ---
RCLONE_BIN = Path(os.environ.get("BACKUP_RCLONE_BINARY", "/opt/rclone-1.74.4/rclone"))
REMOTE = os.environ.get("BACKUP_RCLONE_REMOTE", "backup1tb")
REMOTE_PATH = os.environ.get("BACKUP_RCLONE_PATH", "spread-compacted")
REMOTE_URI = f"{REMOTE}:{REMOTE_PATH}"

# Ограничение для smoke-проверки (None = все окна за lookback).
# Полные ~24h ≈ 5+ GiB скачивания — для быстрой визуализации оставьте 24.
MAX_WINDOWS: int | None = 24
DOWNSAMPLE_EVERY_N = 1  # 1 = все точки; 10 = каждая 10-я (ускоряет plot)

WINDOW_RE = re.compile(
    r"^spread_(\d{8}T\d{6}Z)_(\d{8}T\d{6}Z)(?:_g\d+)?\.parquet$"
)
TS_FMT = "%Y%m%dT%H%M%SZ"

print(f"BASE_COIN={BASE_COIN} LOOKBACK_HOURS={LOOKBACK_HOURS} CACHE_DIR={CACHE_DIR}")
print(f"REMOTE={REMOTE_URI} MAX_WINDOWS={MAX_WINDOWS}")

## 2. Вспомогательные функции

Имена compacted-файлов — это 5‑минутные UTC-окна. Фильтр по монете делается при чтении (`base_coin == …`), потому что hive-партиций в backup нет.

In [ ]:
def parse_window_name(name: str) -> tuple[datetime, datetime] | None:
    m = WINDOW_RE.match(Path(name).name)
    if not m:
        return None
    start = datetime.strptime(m.group(1), TS_FMT).replace(tzinfo=timezone.utc)
    end = datetime.strptime(m.group(2), TS_FMT).replace(tzinfo=timezone.utc)
    return start, end


def list_local_parquets(dirs: list[Path], pattern: str = "spread_*.parquet") -> list[Path]:
    files: list[Path] = []
    for d in dirs:
        if not d.is_dir():
            continue
        files.extend(sorted(d.glob(pattern)))
    # unique by name (prefer first path)
    by_name: dict[str, Path] = {}
    for p in files:
        by_name.setdefault(p.name, p)
    return sorted(by_name.values(), key=lambda p: p.name)


def rclone_lsl() -> list[tuple[str, int]]:
    """Список (filename, size_bytes) с remote. Read-only."""
    cmd = [
        str(RCLONE_BIN),
        "lsl",
        REMOTE_URI,
        "--max-depth",
        "1",
        "--timeout",
        "60s",
        "--retries",
        "1",
    ]
    out = subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT)
    rows: list[tuple[str, int]] = []
    for line in out.splitlines():
        line = line.strip()
        if not line or not line.endswith(".parquet"):
            continue
        parts = line.split()
        # rclone lsl: SIZE YYYY-MM-DD HH:MM:SS.nanoseconds NAME
        if len(parts) < 4:
            continue
        size = int(parts[0])
        name = parts[-1]
        if parse_window_name(name):
            rows.append((name, size))
    return rows


def select_windows(
    names: list[str],
    *,
    start: datetime,
    end: datetime,
    max_windows: int | None,
) -> list[str]:
    picked: list[tuple[datetime, str]] = []
    for name in names:
        bounds = parse_window_name(name)
        if not bounds:
            continue
        w_start, w_end = bounds
        # пересечение с [start, end)
        if w_end <= start or w_start >= end:
            continue
        picked.append((w_start, name))
    picked.sort(key=lambda x: x[0])
    names_sorted = [n for _, n in picked]
    if max_windows is not None and len(names_sorted) > max_windows:
        if max_windows <= 1:
            names_sorted = [names_sorted[0], names_sorted[-1]][:max_windows]
        else:
            # равномерно проредить по времени (smoke), сохранив края
            idx = [
                round(i * (len(names_sorted) - 1) / (max_windows - 1))
                for i in range(max_windows)
            ]
            names_sorted = [names_sorted[i] for i in sorted(set(idx))]
    return names_sorted


def rclone_copy_files(names: list[str], dest: Path) -> list[Path]:
    dest.mkdir(parents=True, exist_ok=True)
    copied: list[Path] = []
    for name in names:
        target = dest / name
        if target.is_file() and target.stat().st_size > 0:
            copied.append(target)
            continue
        cmd = [
            str(RCLONE_BIN),
            "copyto",
            f"{REMOTE_URI}/{name}",
            str(target),
            "--timeout",
            "180s",
            "--retries",
            "3",
            "--low-level-retries",
            "10",
            "--sftp-concurrency",
            "8",
            "--sftp-chunk-size",
            "128k",
        ]
        print("rclone copyto", name, "…")
        subprocess.check_call(cmd)
        copied.append(target)
    return copied


def infer_time_bounds_from_names(names: list[str]) -> tuple[datetime, datetime]:
    ends: list[datetime] = []
    starts: list[datetime] = []
    for name in names:
        b = parse_window_name(name)
        if b:
            starts.append(b[0])
            ends.append(b[1])
    if not ends:
        raise RuntimeError("Не удалось распарсить имена окон")
    data_end = max(ends)
    data_start = min(starts)
    return data_start, data_end


print("helpers ready")

## 3. Скачать окна с backup (VPS) или использовать локальный кэш

**VPS (предпочтительно):** ячейка ниже сделает `rclone lsl` + `copyto` выбранных окон в `CACHE_DIR`.

**Mac:** если rclone к `backup1tb` нет, на VPS выполните copy в каталог, затем `scp`:

```bash
# на VPS
mkdir -p /tmp/spread_backup_cache
/opt/rclone-1.74.4/rclone copy backup1tb:spread-compacted /tmp/spread_backup_cache \
  --include 'spread_20260804T*.parquet' \
  --timeout 120s --retries 2 --sftp-concurrency 8 --sftp-chunk-size 128k

# на Mac (пример)
mkdir -p /tmp/spread_backup_cache
scp 'root@38.244.198.42:/tmp/spread_backup_cache/spread_20260804T11*.parquet' /tmp/spread_backup_cache/
```

Затем в конфиге выше: `CACHE_DIR = Path("/tmp/spread_backup_cache")`, `MAX_WINDOWS` можно не трогать — ячейка чтения возьмёт уже лежащие файлы.

Поставьте `SKIP_DOWNLOAD = True`, если кэш уже готов.

In [ ]:
SKIP_DOWNLOAD = False  # True = только локальный CACHE_DIR / EXTRA_LOCAL_DIRS / LOCAL_PARQUET_GLOB

local_candidates = list_local_parquets([CACHE_DIR, *EXTRA_LOCAL_DIRS])
if LOCAL_PARQUET_GLOB:
    from glob import glob as _glob

    by_name = {p.name: p for p in local_candidates}
    for p in (Path(x) for x in _glob(LOCAL_PARQUET_GLOB)):
        by_name.setdefault(p.name, p)
    local_candidates = sorted(by_name.values(), key=lambda p: p.name)

remote_names: list[str] = []
if not SKIP_DOWNLOAD and RCLONE_BIN.is_file():
    try:
        remote_listing = rclone_lsl()
        remote_names = [n for n, _ in remote_listing]
        print(f"remote files: {len(remote_names)}")
    except Exception as exc:
        print(f"rclone lsl недоступен ({exc}); используем локальный кэш")
else:
    print("SKIP_DOWNLOAD или rclone binary отсутствует — только локальные файлы")

all_names = sorted(set(remote_names) | {p.name for p in local_candidates})
if not all_names:
    raise RuntimeError(
        "Нет parquet для анализа. Скачайте окна в CACHE_DIR или запустите на VPS с rclone."
    )

data_start_all, data_end_all = infer_time_bounds_from_names(all_names)
end = END_UTC or data_end_all
start = START_UTC or (end - timedelta(hours=LOOKBACK_HOURS))
if start < data_start_all:
    start = data_start_all

wanted = select_windows(all_names, start=start, end=end, max_windows=MAX_WINDOWS)
print(f"data coverage on listing: {data_start_all} .. {data_end_all}")
print(f"selected window: {start} .. {end}")
print(f"selected files: {len(wanted)}")
for n in wanted[:3]:
    print("  …", n)
if len(wanted) > 3:
    print("  …")
    print("  …", wanted[-1])

local_by_name = {p.name: p for p in local_candidates}
need_download = [n for n in wanted if n not in local_by_name]

if need_download and not SKIP_DOWNLOAD and RCLONE_BIN.is_file():
    print(f"downloading {len(need_download)} files → {CACHE_DIR}")
    rclone_copy_files(need_download, CACHE_DIR)
    local_candidates = list_local_parquets([CACHE_DIR, *EXTRA_LOCAL_DIRS])
    local_by_name = {p.name: p for p in local_candidates}
elif need_download:
    print(
        f"WARNING: {len(need_download)} файлов нет локально и download пропущен. "
        f"Будут использованы только доступные."
    )

parquet_paths = [local_by_name[n] for n in wanted if n in local_by_name]
if not parquet_paths:
    raise RuntimeError("После отбора не осталось локальных parquet — скачайте кэш.")
print(f"ready to read: {len(parquet_paths)} files")
approx_bytes = sum(p.stat().st_size for p in parquet_paths)
print(f"cache size ≈ {approx_bytes / (1024**3):.3f} GiB")

## 4. Загрузка монеты и sanity-таблица

Читаем только нужные колонки + filter `base_coin`. Если `spread_*` вдруг отсутствуют (lean), считаем из L1:

- `spread_long = (bybit_bid − okx_ask) / bybit_bid × 100`
- `spread_short = (okx_bid − bybit_ask) / okx_bid × 100`

Latency (v1 canary): `okx_latency_ms`, `bybit_latency_ms`, `max_latency_ms` — обычно `local_recv − exchange_ts`. Если колонок нет (lean), график latency пропустится.

In [ ]:
CORE_COLS = [
    "event_dt",
    "event_local_ts_ms",
    "base_coin",
    "spread_long",
    "spread_short",
    "okx_latency_ms",
    "bybit_latency_ms",
    "max_latency_ms",
]
L1_COLS = [
    "okx_bid_price",
    "okx_ask_price",
    "bybit_bid_price",
    "bybit_ask_price",
]
LATENCY_COLS = [
    "okx_latency_ms",
    "bybit_latency_ms",
    "max_latency_ms",
]


def schema_names(path: Path) -> set[str]:
    return set(pq.ParquetFile(path).schema_arrow.names)


def ensure_spreads(df: pd.DataFrame) -> pd.DataFrame:
    if "spread_long" in df.columns and "spread_short" in df.columns:
        return df
    need = set(L1_COLS)
    if not need.issubset(df.columns):
        missing = need - set(df.columns)
        raise RuntimeError(f"Нет spread_* и нет L1 для derive: {sorted(missing)}")
    df = df.copy()
    df["spread_long"] = (
        (df["bybit_bid_price"] - df["okx_ask_price"]) * 100.0 / df["bybit_bid_price"]
    )
    df["spread_short"] = (
        (df["okx_bid_price"] - df["bybit_ask_price"]) * 100.0 / df["okx_bid_price"]
    )
    return df


def read_coin_frames(paths: list[Path], coin: str) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    filt = [("base_coin", "==", coin)]
    for path in paths:
        names = schema_names(path)
        cols = [c for c in CORE_COLS if c in names]
        if "spread_long" not in names or "spread_short" not in names:
            cols = list(dict.fromkeys(cols + [c for c in L1_COLS if c in names]))
        if "event_dt" not in names and "event_local_ts_ms" not in names:
            raise RuntimeError(f"{path.name}: нет event_dt / event_local_ts_ms")
        table = pq.read_table(path, columns=cols, filters=filt)
        if table.num_rows == 0:
            continue
        frame = table.to_pandas()
        if "event_dt" not in frame.columns:
            frame["event_dt"] = pd.to_datetime(
                frame["event_local_ts_ms"], unit="ms", utc=True
            )
        else:
            frame["event_dt"] = pd.to_datetime(frame["event_dt"], utc=True)
        frame = ensure_spreads(frame)
        frames.append(frame)
    if not frames:
        raise RuntimeError(
            f"Монета {coin!r} не найдена в {len(paths)} файлах. "
            f"Проверьте BASE_COIN / bybit_okx_universe.csv."
        )
    out = pd.concat(frames, ignore_index=True)
    out = out.sort_values("event_dt").drop_duplicates(
        subset=["event_dt", "spread_long", "spread_short"], keep="last"
    )
    return out.reset_index(drop=True)


df = read_coin_frames(parquet_paths, BASE_COIN)
if DOWNSAMPLE_EVERY_N > 1:
    df_plot = df.iloc[::DOWNSAMPLE_EVERY_N].copy()
else:
    df_plot = df

latency_present = [c for c in LATENCY_COLS if c in df.columns]
row = {
    "base_coin": BASE_COIN,
    "rows": len(df),
    "files": len(parquet_paths),
    "t_min": df["event_dt"].min(),
    "t_max": df["event_dt"].max(),
    "null_spread_long_pct": 100.0 * df["spread_long"].isna().mean(),
    "null_spread_short_pct": 100.0 * df["spread_short"].isna().mean(),
    "spread_long_min": df["spread_long"].min(),
    "spread_long_max": df["spread_long"].max(),
    "spread_short_min": df["spread_short"].min(),
    "spread_short_max": df["spread_short"].max(),
    "latency_cols": ",".join(latency_present) if latency_present else "(нет)",
}
for col in LATENCY_COLS:
    if col in df.columns:
        s = df[col].dropna()
        row[f"{col}_p50"] = float(s.quantile(0.50)) if len(s) else None
        row[f"{col}_p95"] = float(s.quantile(0.95)) if len(s) else None
        row[f"{col}_max"] = float(s.max()) if len(s) else None
sanity = pd.DataFrame([row])
sanity

## 5. График — на что смотреть

Визуальный чеклист:

1. **Непрерывность** — есть ли длинные дыры (минуты/часы без точек) внутри окна canary?
2. **Масштаб** — спред в процентах; типичные значения около нуля ± доли процента, не «тысячи».
3. **Согласованность long/short** — обычно зеркально-похожий шум; simultaneous spike в обе стороны на сотни % — подозрение на битый L1 / деление.
4. **Разрывы на границах окон** — compacted файлы стыкуются по 5 мин; резкий jump ровно на границе окна может быть артефактом рынка или стыка, стоит сверить соседние файлы.
5. **Null / NaN** — если серия обрывается, смотрите `null_*_pct` в таблице выше.
6. **Latency** (следующая секция) — типично десятки ms; секунды/минуты или отрицательные значения — повод копать clock skew / stale quotes.

Это **не** доказательство прибыльности стратегии — только проверка, что backup читается и спред/latency выглядят правдоподобно.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_plot["event_dt"], df_plot["spread_long"], label="spread_long", linewidth=0.8)
ax.plot(df_plot["event_dt"], df_plot["spread_short"], label="spread_short", linewidth=0.8, alpha=0.85)
ax.axhline(0.0, color="gray", linewidth=0.6, linestyle="--")
ax.set_title(
    f"Спред OKX↔Bybit с backup — {BASE_COIN} "
    f"({df['event_dt'].min()} … {df['event_dt'].max()} UTC)"
)
ax.set_xlabel("Время события (UTC)")
ax.set_ylabel("Спред, %")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5b. Latency — на что смотреть

Колонки v1: `okx_latency_ms`, `bybit_latency_ms`, `max_latency_ms`.

1. **Порядок величины** — обычно десятки ms (см. p50/p95 в sanity).
2. **Спайки** — короткие выбросы на сотни ms допустимы; устойчивый рост до секунд — деградация WS/сети.
3. **Отрицательные** — подозрение на рассинхрон часов или неверный exchange ts.
4. **Дыры вместе со спредом** — если latency и спред пропадают одновременно, проблема скорее в ingest/окне, не в формуле спреда.

In [ ]:
present = [c for c in LATENCY_COLS if c in df_plot.columns]
latency_summary = None
if not present:
    print("Колонок latency нет в данных (lean schema?) — график пропущен.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

    for col in present:
        axes[0].plot(
            df_plot["event_dt"],
            df_plot[col],
            label=col,
            linewidth=0.7,
            alpha=0.9,
        )
    axes[0].set_ylabel("Latency, ms")
    axes[0].set_title(
        f"Latency OKX/Bybit с backup — {BASE_COIN} "
        f"({df['event_dt'].min()} … {df['event_dt'].max()} UTC)"
    )
    axes[0].legend(loc="upper right")
    axes[0].grid(True, alpha=0.3)

    # Нижний subplot: zoom по p99.5, чтобы спайки не схлопывали шкалу
    series_for_ylim = (
        df_plot["max_latency_ms"] if "max_latency_ms" in present else df_plot[present[0]]
    )
    hi = float(series_for_ylim.quantile(0.995)) if series_for_ylim.notna().any() else None
    for col in present:
        axes[1].plot(
            df_plot["event_dt"],
            df_plot[col],
            label=col,
            linewidth=0.7,
            alpha=0.9,
        )
    if hi is not None and hi > 0:
        axes[1].set_ylim(bottom=min(0.0, float(series_for_ylim.min())), top=hi * 1.05)
    axes[1].set_xlabel("Время события (UTC)")
    axes[1].set_ylabel("Latency, ms (zoom p99.5)")
    axes[1].legend(loc="upper right")
    axes[1].grid(True, alpha=0.3)

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

    summary_rows = []
    for col in present:
        s = df[col].dropna()
        summary_rows.append(
            {
                "col": col,
                "n": len(s),
                "p50": s.quantile(0.50),
                "p95": s.quantile(0.95),
                "p99": s.quantile(0.99),
                "max": s.max(),
                "neg_pct": 100.0 * (s < 0).mean(),
            }
        )
    latency_summary = pd.DataFrame(summary_rows)

latency_summary

## 6. Опционально: список монет в одном окне

Если `BASE_COIN` пустой — посмотрите, какие монеты реально есть в первом файле кэша.

In [ ]:
sample = parquet_paths[-1]
coins = (
    pq.read_table(sample, columns=["base_coin"])
    .column("base_coin")
    .to_pylist()
)
uniq = sorted(set(coins))
print(f"file={sample.name} rows={len(coins)} unique_coins={len(uniq)}")
print("has 0G", "0G" in uniq, "| has BTC", "BTC" in uniq)
print("sample:", ", ".join(uniq[:20]), ("…" if len(uniq) > 20 else ""))